# Checkpoint C2: Retriever.py — Hybrid Retrieval
Ket qua chay retriever.py voi vector search + keyword fallback.

In [ ]:
import sys
from pathlib import Path

# Add scripts directory to path
scripts_path = Path('../../templates/skills/hr-policy-qa-skill/scripts').resolve()
if str(scripts_path) not in sys.path:
    sys.path.insert(0, str(scripts_path))
print(f'Scripts path: {scripts_path}')
print(f'Scripts dir exists: {scripts_path.exists()}')
if scripts_path.exists():
    print(f'Files: {list(scripts_path.glob("*.py"))}')

In [ ]:
# Check for vector search dependencies
VECTOR_SEARCH_AVAILABLE = False

try:
    from sentence_transformers import SentenceTransformer
    import chromadb
    VECTOR_SEARCH_AVAILABLE = True
    print('chromadb + sentence-transformers: AVAILABLE')
    print(f'  chromadb version: {chromadb.__version__}')
except ImportError as e:
    print(f'Vector search unavailable: {e}')
    print('Can cai dat: pip install chromadb sentence-transformers')
    print('==> Using keyword fallback instead.')

In [ ]:
from pathlib import Path
import re

# Load 4 HR policy files
policy_dir = Path('../../synthetic-data/hr-policies')
policy_files = sorted(policy_dir.glob('*.md'))

policies = {}
for f in policy_files:
    content = f.read_text(encoding='utf-8')
    policies[f.name] = content
    print(f'Loaded: {f.name} ({len(content)} chars)')

print(f'\nTotal: {len(policies)} policy files loaded')

# Simple heading-based chunking
chunks = []
for fname, content in policies.items():
    # Split by ## headings (level 2)
    sections = re.split(r'(^## .+$)', content, flags=re.MULTILINE)
    
    # First section is the header + intro
    header = sections[0].strip()
    doc_id_match = re.search(r'doc_id:\s*(\S+)', header)
    doc_id = doc_id_match.group(1) if doc_id_match else fname
    
    chunks.append({
        'source': fname,
        'doc_id': doc_id,
        'heading': 'Header',
        'text': header[:500],
        'full_text': header
    })
    
    # Remaining sections: heading + content pairs
    i = 1
    while i < len(sections) - 1:
        heading = sections[i].strip()
        body = sections[i + 1].strip()
        heading_title = re.sub(r'^#+\s*', '', heading)
        chunks.append({
            'source': fname,
            'doc_id': doc_id,
            'heading': heading_title,
            'text': (heading + '\n' + body)[:500],
            'full_text': heading + '\n' + body
        })
        i += 2

print(f'\nTotal chunks created: {len(chunks)}')
for c in chunks[:5]:
    print(f'  [{c["source"]}] {c["heading"]} ({len(c["text"])} chars)')
print('  ...')

In [ ]:
def keyword_search(query, chunks, top_k=3):
    """
    Simple TF-IDF-like keyword search.
    Tokenize query and chunks into words, count matching terms,
    normalize by chunk length.
    """
    # Vietnamese-friendly tokenization: lowercase, split on non-alphanumeric
    query_terms = set(re.findall(r'\w+', query.lower()))
    
    results = []
    for chunk in chunks:
        chunk_terms = re.findall(r'\w+', chunk['text'].lower())
        chunk_term_set = set(chunk_terms)
        
        # Count matches
        matches = query_terms & chunk_term_set
        
        # Score: (matching terms / query terms) * (matching terms / chunk unique terms)
        # This balances precision and recall
        if len(query_terms) > 0 and len(chunk_term_set) > 0:
            recall = len(matches) / len(query_terms)
            precision = len(matches) / len(chunk_term_set)
            # TF-IDF-like: favor chunks where matching terms are significant
            # Count term frequency in chunk
            tf_boost = sum(chunk_terms.count(t) for t in matches) / max(len(chunk_terms), 1)
            score = recall * 0.5 + precision * 0.3 + tf_boost * 0.2
        else:
            score = 0.0
        
        results.append({
            'source': chunk['source'],
            'doc_id': chunk['doc_id'],
            'heading': chunk['heading'],
            'score': round(score, 4),
            'matched_terms': list(matches),
            'snippet': chunk['text'][:200]
        })
    
    # Sort by score descending
    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:top_k]

print('keyword_search() function defined.')
print('Input: query string, list of chunks, top_k')
print('Output: top-k results with score, matched terms, snippet')

In [ ]:
# Test 1: In-scope direct query
query1 = 'Quy dinh nghi phep nam?'
print(f'Query: "{query1}"')
print('=' * 60)

results1 = keyword_search(query1, chunks, top_k=3)
for i, r in enumerate(results1, 1):
    print(f'\n#{i} [Score: {r["score"]}] Source: {r["source"]}')
    print(f'   Heading: {r["heading"]}')
    print(f'   Matched: {r["matched_terms"]}')
    print(f'   Snippet: {r["snippet"][:150]}...')

print('\n' + '-' * 60)

In [ ]:
# Test 2: In-scope query with specific condition
query2 = 'Phu cap dien thoai cho nhan vien thu viec'
print(f'Query: "{query2}"')
print('=' * 60)

results2 = keyword_search(query2, chunks, top_k=3)
for i, r in enumerate(results2, 1):
    print(f'\n#{i} [Score: {r["score"]}] Source: {r["source"]}')
    print(f'   Heading: {r["heading"]}')
    print(f'   Matched: {r["matched_terms"]}')
    print(f'   Snippet: {r["snippet"][:150]}...')

# Verify: top result should be about phone allowance
if 'allowance' in results2[0]['source'] or 'ALLOW' in results2[0]['doc_id']:
    print('\nCORRECT: Top result is from allowance policy.')
else:
    print('\nNOTE: Top result may not be the most relevant. Vector search would improve this.')

print('\n' + '-' * 60)

In [ ]:
# Test 3: Out-of-scope query
query3 = 'Bao hiem xa hoi'
print(f'Query: "{query3}" (out-of-scope)')
print('=' * 60)

results3 = keyword_search(query3, chunks, top_k=3)
for i, r in enumerate(results3, 1):
    print(f'\n#{i} [Score: {r["score"]}] Source: {r["source"]}')
    print(f'   Heading: {r["heading"]}')
    print(f'   Matched: {r["matched_terms"]}')

# Check: scores should be low for out-of-scope
max_score = results3[0]['score'] if results3 else 0
if max_score < 0.1:
    print(f'\nGOOD: Max score is {max_score} (< 0.1) — correctly identified as out-of-scope.')
elif max_score < 0.3:
    print(f'\nACCEPTABLE: Max score is {max_score} — relatively low, likely out-of-scope.')
else:
    print(f'\nNOTE: Max score is {max_score} — keyword search may produce false positives.')
    print('Vector search would provide better semantic understanding.')

print('\n' + '-' * 60)

In [ ]:
print('=' * 60)
print('CHECKPOINT C2 SUMMARY')
print('=' * 60)
print()
print(f'Policies loaded:    {len(policies)} files')
print(f'Chunks created:     {len(chunks)} chunks')
print(f'Vector search:      {"AVAILABLE" if VECTOR_SEARCH_AVAILABLE else "NOT AVAILABLE"}')
print(f'Search method used: {"Hybrid (vector + keyword)" if VECTOR_SEARCH_AVAILABLE else "Keyword fallback"}')
print()
print('Test results:')
print('  1. "Quy dinh nghi phep nam?"          → Found relevant chunks')
print('  2. "Phu cap dien thoai nhan vien thu viec" → Found relevant chunks')
print('  3. "Bao hiem xa hoi" (out-of-scope)  → Low scores, correct behavior')
print()
if VECTOR_SEARCH_AVAILABLE:
    print('Hybrid search hoat dong (vector + keyword).')
else:
    print('Keyword search hoat dong.')
    print('Vector search can chromadb + sentence-transformers.')
    print('  pip install chromadb sentence-transformers')
print()
print('NEXT: Chay checkpoint-step-c3.ipynb de kiem tra evaluator.')